# Softmax logistic regression, PCA and preprocessing methods

## Goal of this notebook
This is the first *modelling* notebook. It builds our baseline classifier and uses it to understand two main questions.

1. **Is PCA a good idea for this dataset?** By analysing the data it is possible to understand the effectiveness of PCA, whether it is worth using it and how many features should be kept.
2. **Which preprocessing works best?** By keeping the model fixed (softmax logistic regression) and changing only the preprocessing, it can be assessed whether a preprocessing method is effective or not by looking at the scores

## Decisions taken
- **5 AAMI classes** instead of the 15 original labels (some of the original labels have too few data, class 9 doesn't even have any validation or test data).
- **Scores: F1 and recall, macro-averaged**. Recall is the priority (missing an arrhythmia is the expensive error), but we select models with F1, so that a model cannot reach high recall by flagging every beat as an arrhythmia.

## Notebook chapters
| Section | What we do |
|---|---|
| 1 | Load train / validation / test |
| 2 | Remap the 15 labels to the 5 AAMI classes |
| 3 | Evaluation tools (metrics + confusion matrix) |
| 4 | Trivial baseline ("always N"): the score every model must beat |
| 5 | PCA analysis: variance, components, per-class reconstruction error |
| 6 | Softmax logistic regression: model structure |
| 7 | Preprocessing methods P0–P3: tuning on validation and comparison |
| 8 | Conclusions |

## 0. Imports and constants

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_preprocessing.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or a notebook subfolder.")

DATA_DIR = PROJECT_ROOT / "data"
if not all((DATA_DIR / name).is_file() for name in (
    "train_data.csv", "val_data.csv", "test_data.csv"
)):
    raise FileNotFoundError("Dataset missing: see data/README.md for the required CSV files.")


# Names of the 15 original labels
ORIGINAL_NAMES = ["N", "/", "L", "R", "e", "j", "A", "a", "J", "S", "E", "F", "V", "f", "Q"]

# Names of the 5 AAMI classes
AAMI_NAMES = ["N", "S", "V", "F", "Q"]
N_CLASSES = len(AAMI_NAMES)

## 1. Load the data

In [ ]:
# Load one split (train, validation or test) from a csv file.
# The labels are the original ones, 0 to 14.

def load_split(path):
    data = pd.read_csv(path, header=None)

    X = data.iloc[:, :-1].to_numpy(dtype=float)  
    y = data.iloc[:, -1].to_numpy().astype(int) 

    return X, y

In [ ]:
X_train, y_train_orig = load_split(DATA_DIR / "train_data.csv")
X_val, y_val_orig = load_split(DATA_DIR / "val_data.csv")
X_test, y_test_orig = load_split(DATA_DIR / "test_data.csv")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train_orig.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val_orig.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test_orig.shape)

## 2. Remap the labels to the 5 AAMI classes

The AAMI standard groups the beat types into 5 classes. We give them new numeric labels 0..4:

| New label | AAMI class | Original labels (numeric) |
|---|---|---|
| 0 | **N** – normal-like | N (0), L (2), R (3), e (4), j (5) |
| 1 | **S** – supraventricular ectopic | A (6), a (7), J (8), S (9) |
| 2 | **V** – ventricular ectopic | E (10), V (12) |
| 3 | **F** – fusion | F (11) |
| 4 | **Q** – paced / unknown | / (1), f (13), Q (14) |

The remapping has been done via a lookup array, an array of length 15 where position `i` contains the new label of the original label `i`.

In [ ]:
# Original numeric label -> AAMI numeric label
ORIGINAL_TO_AAMI = {
    0: 0,    # N -> N
    1: 4,    # / -> Q
    2: 0,    # L -> N
    3: 0,    # R -> N
    4: 0,    # e -> N
    5: 0,    # j -> N
    6: 1,    # A -> S
    7: 1,    # a -> S
    8: 1,    # J -> S
    9: 1,    # S -> S
    10: 2,   # E -> V
    11: 3,   # F -> F
    12: 2,   # V -> V
    13: 4,   # f -> Q
    14: 4,   # Q -> Q
}

# Convert an array of original labels (0..14) into AAMI labels (0..4).
def to_aami(y_orig):
    
    lookup = np.array([ORIGINAL_TO_AAMI[i] for i in range(len(ORIGINAL_NAMES))])

    return lookup[y_orig]

In [ ]:
y_train = to_aami(y_train_orig)
y_val = to_aami(y_val_orig)
y_test = to_aami(y_test_orig)

print("y_train labels: ",np.bincount(y_train, minlength=N_CLASSES))
print("y_val labels: ", np.bincount(y_val, minlength=N_CLASSES))   
print("y_test labels: ", np.bincount(y_test, minlength=N_CLASSES))


## 3. Evaluation tools

### Why macro F1 and macro recall
About 83 % of the beats are N. Metrics that weight classes by their size (accuracy, *weighted* F1) are dominated by N and can look good even when the rare classes are always wrong.

If our model always predicted class N, it would have 83% accuracy even though it is a terrible model.

### Why we also compute the scores on the training set
Comparing train and validation tells us whether a model underfits (both low) or overfits (train high, validation low). It can be seen in this notebook that the model underfits (classes are not linearly separable).

### Confusion matrix normalised by row
If every row is divided by its sum, the diagonal contains the per-class recall. With raw counts the N row would hide everything else.

In [ ]:
# Compute the scores we use to compare models.
def evaluate(y_true, y_pred):
    labels = list(range(N_CLASSES))

    # Per-class values: one array of 5 numbers each
    recall_per_class = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)

    # Macro = plain mean over the 5 classes
    scores = {
        "f1_macro": f1_per_class.mean(),
        "recall_macro": recall_per_class.mean(),
    }
    for i in range(N_CLASSES):
        name = AAMI_NAMES[i]
        scores["recall_" + name] = recall_per_class[i]
        scores["f1_" + name] = f1_per_class[i]

    return scores

# Plot the confusion matrix
# normalize=True: each row is divided by its sum, so the diagonal is the per-class recall
# (needed because some classes are very rare).
def plot_confusion_matrix(y_true, y_pred, title="", normalize=True):
    conf_mat = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES), normalize="true" if normalize else None)
    conf_disp = ConfusionMatrixDisplay(conf_mat, display_labels=AAMI_NAMES)
    conf_disp.plot(cmap=plt.cm.Blues, values_format=".2f" if normalize else "d")
    conf_disp.ax_.set_title(title)
    plt.show()

## 4. Trivial baseline: always predict N

Before training a real model we need the score of a model that learns nothing. `DummyClassifier(strategy="most_frequent")` always predicts the most common class of the training set, which is N.

Every later model must clearly beat this on **macro F1**, otherwise it has learned nothing useful about arrhythmias (even if its accuracy looks high).

In [ ]:
# Trivial baseline: always predict the most frequent class of the training set (N).
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_val_pred_dummy = dummy.predict(X_val)
scores_dummy = evaluate(y_val, y_val_pred_dummy)
display(pd.Series(scores_dummy).round(3))

plot_confusion_matrix(y_val, y_val_pred_dummy, title="Trivial baseline (always N) - validation")

## 5. PCA analysis: would PCA help?

PCA finds the directions of largest variance in the data (the eigenvectors of the covariance matrix) and we can choose how many of them (k) to keep.

PCA compresses well when features have a lot of redundancy, which is the case for our data. The biggest problem with PCA is that it's *unsupervised*, so it keeps the directions of large variance, not those of a good class separation.

The next cells are going to answer these questions:

- How many components hold 90 / 95 / 99 % of the variance?
- What do the components represent (signal shape or noise)?
- Does truncating hurt the rare classes more than N?

These are indications only. The **final verdict comes in Section 7**, where PCA is placed inside the classifier and judged by validation macro F1.

### 5.1 Fit PCA on the training set

PCA is fitted with all 250 components once, on training set only.

In [ ]:
# Fit PCA with all components on the training data.
def fit_full_pca(X_train):
    pca = PCA(n_components=None, random_state=RANDOM_STATE)
    pca.fit(X_train)
    return pca

In [ ]:
pca_full = fit_full_pca(X_train)

### 5.2 Cumulative explained variance

Given a target value of explained variance, the function returns the number of components needed to reach it. Here, 17 components (out of 250) are enough to explain 99% of the variance, which shows that the features are very redundant.

This looks very promising, but it hides a problem. N makes up about 83% of the dataset, so the 99% is mostly variance of normal beats. The remaining 1% may not be spread evenly across the classes: it could contain exactly the details that distinguish the rarer ones. This is studied in Section 5.4.

In [ ]:
# Plot the cumulative explained variance up to the k-th component

def plot_explained_variance(pca, thresholds=(0.90, 0.95, 0.99)):
    
    # cumulative_variance[i] = variance kept by the first i+1 components
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    n_comp = len(cumulative_variance)

    # searchsorted gives the first position where cumulative_variance >= t
    k_for_threshold = {}
    for t in thresholds:
        k_for_threshold[t] = int(np.searchsorted(cumulative_variance, t) + 1)

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, n_comp + 1), cumulative_variance, color="black")
    for t in thresholds:
        k = k_for_threshold[t]
        line = plt.axvline(k, linestyle="--", label=f"{t:.0%} -> k = {k}")
        plt.axhline(t, linestyle=":", color=line.get_color())
    plt.xscale("log")
    plt.xlabel("Number of components k")
    plt.ylabel("Cumulative explained variance")
    plt.title("PCA on the training set: cumulative explained variance")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return k_for_threshold

In [ ]:
k_for_threshold = plot_explained_variance(pca_full)
print(k_for_threshold)

### 5.3 What do the components look like?

One of the uses of PCA is denoising. This is mostly useful for models that overfit, since they can end up fitting the noise instead of the real structure of the data. Logistic regression underfits here, but the non-linear models used later (SVM, neural network) might not, so it is worth checking what the components contain.

Each component is a vector of 250 weights, one per time sample, so it can be plotted as a waveform. The plot shows the first 6 components and the last 3.

In [ ]:
# plots components as waveforms
def plot_components(pca, n=6):
    n_comp = len(pca.components_)
    n_rows = n // 3 + 1

    fig, axes = plt.subplots(n_rows, 3, figsize=(12, 7))

    # First n components, 3 per row
    for i in range(n):
        row = i // 3  #integer division
        col = i % 3
        axes[row, col].plot(pca.components_[i])
        axes[row, col].set_title(f"PC{i + 1} ({pca.explained_variance_ratio_[i]:.2%} of variance)")

    # Last row: the last 3 components
    for col in range(3):
        i = n_comp - 3 + col
        axes[n_rows - 1, col].plot(pca.components_[i])
        axes[n_rows - 1, col].set_title(f"PC{i + 1} ({pca.explained_variance_ratio_[i]:.2%} of variance)")

    fig.suptitle("Principal components as waveforms (last row: last 3 components)")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_components(pca_full)

### 5.4 Per-class reconstruction error: does PCA hurt the rare classes?

This is the most important check. PCA with $k$ components maximises the variance kept over the whole dataset. Since that variance is dominated by N, the $k$ components may describe N beats very well and the rare classes worse.

To check this, for each value of $k$:

1) only the first $k$ components are kept: each beat is projected on them and then mapped back to 250 samples;
2) the reconstruction error of each beat is computed, i.e. the mean squared difference between the original and the reconstruction;
3) the error is averaged per class.

PCA is fitted on the training set and the errors are computed on the validation beats. The values of $k$ are the thresholds from Section 5.2 (5, 8, 17) plus some larger ones (50, 70, 100, 115).

Section 5.2 suggested k = 17, since it keeps 99% of the variance. The tables below show that at k = 17 the reconstruction error of the rare classes is 2 to 3 times larger than that of N. This is why larger values of $k$ were tested as well.

In [ ]:
# Compute the mean reconstruction error per class for different k values.

def reconstruction_error_per_class(X_train, X_eval, y_eval, k_values):
    rows = []  # one dict per k, it becomes one row of the table

    for k in k_values:
        # PCA with only the first k components
        pca = PCA(n_components=k, random_state=RANDOM_STATE)
        pca.fit(X_train)

        # Project on the k components (k numbers per beat) and map back to 250 samples
        X_rec = pca.inverse_transform(pca.transform(X_eval))

        error_per_beat = np.mean((X_eval - X_rec) ** 2, axis=1) 

        # Mean error of the beats of each class
        row = {}
        for c in range(N_CLASSES):
            row[AAMI_NAMES[c]] = error_per_beat[y_eval == c].mean()
        rows.append(row)

    errors = pd.DataFrame(rows, index=k_values)
    errors.index.name = "k"
    return errors

In [ ]:
errors = reconstruction_error_per_class(X_train, X_val, y_val, k_values=sorted(k_for_threshold.values())+ [50,70,100,115])
display(errors)
relative_errors = errors.div(errors["N"], axis=0)
display(relative_errors)

At k = 5 and 8, V and F are reconstructed 3 to 4 times worse than N, and at k = 17 all the rare classes are still 2 to 3 times worse. From k ≈ 50 the ratios of S, V and F are close to 1. Q needs more components (ratio ≈ 2.5 up to k = 70).

PCA can therefore compress the data, but not as much as the variance thresholds suggest.

## 6. Softmax logistic regression

The chosen model for this notebook is *softmax logistic regression*. This is a linear multiclass classifier: it computes one score per class and turns the scores into probabilities that sum to 1. The prediction is the class with the highest probability. 

**Settings in sklearn**

- **L2 penalty**: an extra term added to the cross-entropy loss, it keeps the weights small. `C` is the inverse of the regularisation strength: high C -> weak regularisation, small C -> strong regularisation.
- **`class_weight="balanced"`**: each class contributes the same total weight to the loss, otherwise N (83 % of the beats) would dominate and the model could ignore S and F.
- **`max_iter = 1000`**: maximum number of optimiser iterations.

In [ ]:
# return an unfitted logistic regression model with softmax (multinomial) loss and L2 penalty.
def build_logreg(C, max_iter=1000):

    model = LogisticRegression(
        solver="lbfgs",
        C=C,
        class_weight="balanced",  # misclassifying N costs way less than misclassifying the other classes
        max_iter=max_iter,
        random_state=RANDOM_STATE,
    )
    return model

#### One function that trains and scores any model

In [ ]:
# Fit the model on the training data and evaluate it on train and validation.
def fit_and_score(model, X_tr, y_tr, X_va, y_va):

    model.fit(X_tr, y_tr)

    scores_train = evaluate(y_tr, model.predict(X_tr))
    scores_val = evaluate(y_va, model.predict(X_va))

    result = {}
    for key in scores_train:
        result["train_" + key] = scores_train[key]
    for key in scores_val:
        result["val_" + key] = scores_val[key]

    return result

## 7. Preprocessing methods

In Section 7 we evaluate the softmax logistic regression on the same data with different preprocessing methods. The classifier is always the same (`build_logreg(C)`), only the preprocessing changes.

### The preprocessing methods we compare

| ID | Steps before the classifier | What it tests |
|---|---|---|
| **P0** | none (raw) | reference |
| **P1** | per-beat baseline removal + per-beat amplitude normalisation | removes the baseline wander and the amplitude differences between beats |
| **P2** | `PCA(k)` | compression / denoising |
| **P3** | `PCA(k, whiten=True)` | PCA-sphering |

P2 and P3 use the same PCA as Section 5, so the per-class analysis of 5.4 holds for them.
With `whiten=True`, PCA divides each kept component by its standard deviation $\sqrt{d_n}$, where $d_n$ is the variance of the n-th component: after whitening all the kept components have variance 1.


### 7.1 The per-beat transformations

**Design choices:**
- *Baseline:* the **median** of the beat. Most samples lie on the flat part of the beat, so the median gives its level; the mean would be pulled up or down by the QRS peak.
- *Amplitude:* the **standard deviation** of the beat. It uses all 250 samples, so it is less sensitive than the maximum absolute value to a single very high sample. A tiny epsilon avoids a division by zero.

In [ ]:
# Preprocessing: remove baseline and normalize amplitude of each beat (row).
def remove_baseline(X):

    baseline = np.median(X, axis=1, keepdims=True) 

    X_centered = X - baseline
    return X_centered


def normalize_amplitude(X, eps=1e-8):

    scale = np.std(X, axis=1, keepdims=True) 

    X_scaled = X / (scale + eps)
    return X_scaled

### 7.2 Preprocessing the data of one method

One function applies the steps of every method in the table: given the method ID (and `k` for P2 and P3), it returns the transformed training and validation sets. The grid loop in 7.3 calls it once for every method and every `k`.

The only step that is fitted is PCA: it is fitted on the training set only and then applied to both sets, so no information from the validation set leaks into the model.

In [ ]:
# Apply the preprocessing of method_id (P0-P3) to the training and validation sets.
# k is the number of PCA components, used only by P2 and P3.
def preprocess(method_id, X_tr, X_va, k=None):

    if (method_id == "P2" or method_id == "P3") and k is None:
        raise ValueError(method_id + " needs the number of PCA components k")

    if method_id == "P0":
        X_tr_new = X_tr
        X_va_new = X_va

    elif method_id == "P1":
        # First the baseline, then the amplitude
        X_tr_new = normalize_amplitude(remove_baseline(X_tr))
        X_va_new = normalize_amplitude(remove_baseline(X_va))

    elif method_id == "P2" or method_id == "P3":
        # PCA with k components on the raw data
        # P3 also divides every component by its std
        whiten = (method_id == "P3")
        pca = PCA(n_components=k, whiten=whiten, random_state=RANDOM_STATE)
        pca.fit(X_tr)
        X_tr_new = pca.transform(X_tr)
        X_va_new = pca.transform(X_va)

    else:
        raise ValueError("Unknown method_id: " + method_id)

    return X_tr_new, X_va_new

### 7.3 Tuning strategy

Each method gets its own best hyperparameters. Comparing all of them with the same `C` would favour whichever method happens to like that value.

- **`C` grid:** log-spaced, because its effect is multiplicative (0.001, 0.01, 0.1, …).
- **`k` grid (P2, P3 only):** the values chosen in 5.4 (17 as a reference, then 30, 50, 70, 100).
- **Selection:** highest **validation macro F1**.

In [ ]:
C_GRID = np.logspace(-3, 2, 6)   # 0.001, 0.01, 0.1, 1, 10, 100
K_GRID = [17, 30, 50, 70, 100]   # values chosen in 5.4
METHOD_IDS = ["P0", "P1", "P2", "P3"]

In [ ]:
# Train and score one method for every hyperparameter combination.
# Returns a table with one row per run: 'method', 'C', 'k' + every score of fit_and_score.
def run_grid(method_id, C_grid, k_grid, X_tr, y_tr, X_va, y_va):

    # Only P2 and P3 have a k
    if method_id == "P2" or method_id == "P3":
        k_values = k_grid
    else:
        k_values = [None]

    rows = [] 

    for k in k_values:
        X_tr_new, X_va_new = preprocess(method_id, X_tr, X_va, k)

        for C in C_grid:
            result = fit_and_score(build_logreg(C), X_tr_new, y_tr, X_va_new, y_va)

            # Hyperparameters first, then all the scores
            row = {"method": method_id, "C": C, "k": k}
            for key in result:
                row[key] = result[key]
            rows.append(row)

            print(f"{method_id:4} k={k}  C={C:g}  val_f1_macro={result['val_f1_macro']:.3f}")

    results = pd.DataFrame(rows)
    return results

In [ ]:
# Run the grid of every method and put all the runs in one table
results_per_method = []
for method_id in METHOD_IDS:
    results = run_grid(method_id, C_GRID, K_GRID, X_train, y_train, X_val, y_val)
    results_per_method.append(results)

all_results = pd.concat(results_per_method, ignore_index=True)

display(all_results[["method", "C", "k", "val_f1_macro", "val_recall_macro"]].round(3))

### 7.4 Best configuration of each method

From all runs, keep for every method the row with the highest validation macro F1. This gives one row per method.

How: loop over the method IDs, take the runs of that method, and find the row where `val_f1_macro` is maximal.

In [ ]:
# Return one row per method: the configuration with the best validation macro F1,
# sorted from the best method to the worst.
def best_per_method(results):

    best_rows = []
    for method_id in results["method"].unique():
        runs = results[results["method"] == method_id]

        # Index of the run with the highest validation macro F1
        best_index = runs["val_f1_macro"].idxmax()
        best_rows.append(results.loc[best_index])

    best = pd.DataFrame(best_rows)
    best = best.set_index("method")
    best = best.sort_values("val_f1_macro", ascending=False)
    return best

In [ ]:
best = best_per_method(all_results)
display(best[["C", "k", "val_f1_macro", "val_recall_macro", "train_f1_macro", "train_recall_macro"]].round(3))

### 7.5 Winner method confusion matrix

In this section we rebuild the best method with its best `C` and fit it on the training set, then we plot the *normalized confusion matrix on validation*.  

We also print the `classification_report` for the full per-class precision / recall / F1 table.

The row N is the most important one: it shows how many normal beats are flagged as arrhythmias (remember that the model has class_weight = "balanced").

In [ ]:
# The best method is the first row of best (sorted by val_f1_macro)
winner = best.index[0]
C_best = best.loc[winner, "C"]
k_best = best.loc[winner, "k"]

if pd.isna(k_best):
    k_best = None
else:
    k_best = int(k_best)

print(f"Best method: {winner}, C = {C_best:g}, k = {k_best}")

# Preprocess, then fit a new model on train
X_tr_best, X_va_best = preprocess(winner, X_train, X_val, k_best)
model_best = build_logreg(C_best)
model_best.fit(X_tr_best, y_train)
y_val_pred_best = model_best.predict(X_va_best)

plot_confusion_matrix(y_val, y_val_pred_best, title=f"Best logistic regression ({winner}, C = {C_best:g}) - validation")
print(classification_report(y_val, y_val_pred_best, target_names=AAMI_NAMES, digits=3))

## 8. Conclusions

### Best configuration of each method

| Method | C | k | val macro F1 | val macro recall | train macro F1 |
|---|---|---|---|---|---|
| **P1** baseline + amplitude | 0.1 | – | **0.554** | **0.834** | 0.552 |
| P3 PCA-sphering | 0.001 | 100 | 0.543 | 0.814 | 0.540 |
| P0 raw | 1 | – | 0.532 | 0.814 | 0.534 |
| P2 PCA | 1 | 100 | 0.531 | 0.813 | 0.532 |
| *Trivial baseline (always N)* | | | *0.18* | | |

The differences between methods are small (0.531–0.554). Preprocessing does not change the overall level of the linear model.

### PCA
- **Consistent with Section 5.** With k = 17 (99 % of the variance) the macro F1 drops to 0.474, against 0.532 with all 250 features: the information lost in the last 1 % of the variance matters for the rare classes.
- The score grows with k (17: 0.474, 30: 0.512, 50: 0.524, 70: 0.528, 100: 0.531). About **100 components** are needed to match the full feature set. PCA compresses 250 → 100 features without loss, but it does not improve the score: for a linear model, dropping components can only remove information.
- **Whitening (P3) is slightly better than plain PCA (P2)** at k = 100 (0.543 vs 0.531), and only with strong regularisation (C = 0.001).

### Per-beat normalisation
- **Baseline removal + amplitude normalisation (P1) gives the best method**, but the gain over the raw data is small (+0.022 macro F1 over P0).

### Takeaways for the next models
- A non-linear decision boundary is needed to separate S and F from N.
- If PCA is used (e.g. for the SVM), keep about 100 components and never fewer than 50.

### Model
- **No overfitting, but clear underfitting.** Train and validation macro F1 differ by less than 0.005 for every method. The limit is the linear model, not the amount of data.